In [ ]:
from google.colab import drive
from pathlib import Path
from urllib.request import urlretrieve
import glob
import shutil
import subprocess

url = "https://github.com/HaurylanClaan/projeto_big_data/releases/download/v1.0-dataset/data_base.rar"
pasta = Path("/content/data_base_extraida")
arquivo_rar = Path("/content/data_base.rar")
pasta.mkdir(parents=True, exist_ok=True)

if not any(pasta.glob("*.csv")):
    print("Baixando a base de dados do GitHub Releases...")
    urlretrieve(url, arquivo_rar)

    # Garantir que o unrar está instalado
    if shutil.which("unrar") is None:
        print("Instalando unrar...")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "unrar"], check=True)

    print("Extraindo arquivo RAR...")
    # Extrair usando o unrar (comando 'x' para extrair com caminho completo, '-o+' para sobrescrever)
    subprocess.run(
        ["unrar", "x", "-o+", str(arquivo_rar), str(pasta)],
        check=True
    )

arquivos = sorted(glob.glob(str(pasta / "*.csv")))
if not arquivos:
    raise FileNotFoundError(
        f"Nenhum CSV foi encontrado após extrair o arquivo em {pasta}"
    )

drive.mount("/content/drive")
print("Quantidade de arquivos encontrados:", len(arquivos))

In [ ]:
lista_dfs = []

for arquivo in arquivos:
    df = pd.read_csv(arquivo)
    lista_dfs.append(df)

dados = pd.concat(lista_dfs,
ignore_index=True)

print("Arquivos unidos:",
 len(arquivos))
print("Quantidade de linhas:",
 len(dados))
print("Quantidade de colunas:",
 len(dados.columns))

dados.head()

In [ ]:
# Verificar valores nulos

print("Quantidade de linhas:", len(dados))
print("Quantidade de colunas:", len(dados.columns))

print("\nValores nulos por coluna:")
print(dados.isnull().sum())

In [ ]:
# Definir limite de 95% de nulos para exclusão de colunas
limite_nulos = 0.95
quantidade_limite = len(dados) * limite_nulos

# Filtrar colunas que possuem menos nulos que o limite estabelecido
colunas_antes = dados.shape[1]
colunas_manter = dados.columns[dados.isnull().sum() <= quantidade_limite]

dados = dados[colunas_manter]
colunas_depois = dados.shape[1]

print(f"Colunas originais: {colunas_antes}")
print(f"Colunas mantidas (com até 95% de nulos): {colunas_depois}")
print(f"Colunas removidas por excesso de nulos: {colunas_antes - colunas_depois}")

In [ ]:
# Apenas removemos as colunas com excesso de nulos para o tratamento,
# sem realizar preenchimento artificial que aumente o tamanho da base.

print("Tratamento concluído! Mantivemos apenas as colunas mais significativas.")
print("Quantidade final de colunas:", dados.shape[1])
print("Quantidade final de linhas:", dados.shape[0])

In [ ]:
duplicadas = dados.duplicated().sum()

print("Linhas duplicadas:", duplicadas)

In [ ]:
# Remover linhas duplicadas

dados = dados.drop_duplicates()

print("Linhas após remover duplicatas:", len(dados))
print("Duplicatas restantes:", dados.duplicated().sum())

In [ ]:
# Remover colunas totalmente vazias

antes = len(dados.columns)

dados = dados.dropna(axis=1, how="all")

depois = len(dados.columns)

print("Colunas antes:", antes)
print("Colunas depois:", depois)
print("Colunas removidas:", antes - depois)

In [ ]:
print(dados.dtypes)

In [ ]:
# Salvar base tratada diretamente no ambiente local do Colab

caminho_saida = "/content/base_tratada.csv"

dados.to_csv(caminho_saida, index=False)

print("Base salva com sucesso localmente!")
print(caminho_saida)